In [1]:
"""
Google Colab Ready Training Script for SmolLM-135M
==================================================

This script is optimized for Google Colab with A100 GPU support.

SETUP INSTRUCTIONS FOR GOOGLE COLAB:
====================================

1. Install dependencies:
   !pip install torch transformers numpy

2. Mount Google Drive (for files and checkpoints):
   from google.colab import drive
   drive.mount('/content/drive')

3. Upload your files to Google Drive:
   - marathi_tokens.bin (pre-tokenized binary file)
   - marathi_tokens_meta.json (metadata file)
   - marathi-bpe-tokenizer-49k.json (tokenizer)
   - model.py (model architecture)

4. Update file paths in the configuration section below

5. Run the script:
   !python train_smol.py

   OR paste this entire script into a Colab cell and run it

6. Checkpoints will be saved to Google Drive automatically
"""

"\nGoogle Colab Ready Training Script for SmolLM-135M\n==================================================\n\nThis script is optimized for Google Colab with A100 GPU support.\n\nSETUP INSTRUCTIONS FOR GOOGLE COLAB:\n====================================\n\n1. Install dependencies:\n   !pip install torch transformers numpy\n\n2. Mount Google Drive (for files and checkpoints):\n   from google.colab import drive\n   drive.mount('/content/drive')\n\n3. Upload your files to Google Drive:\n   - marathi_tokens.bin (pre-tokenized binary file)\n   - marathi_tokens_meta.json (metadata file)\n   - marathi-bpe-tokenizer-49k.json (tokenizer)\n   - model.py (model architecture)\n\n4. Update file paths in the configuration section below\n\n5. Run the script:\n   !python train_smol.py\n   \n   OR paste this entire script into a Colab cell and run it\n\n6. Checkpoints will be saved to Google Drive automatically\n"

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast
import os
import sys
import numpy as np
import json
from pathlib import Path
import shutil


In [4]:
# --- Detect if running in Colab ---
IN_COLAB = 'google.colab' in sys.modules
print(IN_COLAB)


True


In [5]:
# Cell 2: Add path and verify
import sys
sys.path.insert(0, '/content/drive/MyDrive/ERA_V4/SmolLM_135M_Marathi')
from model import SmolLM2, SmolLMConfig
print("✅ Ready to train!")

✅ Ready to train!


In [6]:
# --- Import model (handle both Colab and local) ---
if IN_COLAB:
    # In Colab, model.py should be in the same directory or uploaded
    try:
        from model import SmolLM2, SmolLMConfig
    except ImportError:
        print("⚠️  ERROR: model.py not found!")
        print("   Please upload model.py to Colab or ensure it's in the current directory")
        sys.exit(1)
else:
    from model import SmolLM2, SmolLMConfig


In [7]:
# --- Configuration ---

if IN_COLAB:
    # Colab default working directory
    BASE_DIR = Path('/content')

    # OPTION 1: Files in Colab's /content directory (uploaded via files.upload())
    # TOKENS_BIN_FILE = BASE_DIR / "marathi_tokens.bin"
    # TOKENS_META_FILE = BASE_DIR / "marathi_tokens_meta.json"
    # TOKENIZER_PATH = BASE_DIR / "tokenizer" / "marathi-bpe-tokenizer-49k.json"

    # OPTION 2: Files in Google Drive (RECOMMENDED)
    # First mount drive: drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/ERA_V4/SmolLM_135M_Marathi')

    # Update these paths to match your Google Drive structure:
    TOKENS_BIN_FILE = DRIVE_DIR / "pre_process" / "marathi_tokens.bin"  # Update this path
    TOKENS_META_FILE = DRIVE_DIR / "pre_process" / "marathi_tokens_meta.json"  # Update this path
    TOKENIZER_PATH = DRIVE_DIR / "tokenizer" / "marathi-bpe-tokenizer-49k.json"  # Update this path

    # Checkpoints will be saved to Drive (so they persist after session ends)
    CHECKPOINT_DIR = DRIVE_DIR / "checkpoints" / "smollm_135m"

else:
    # Local machine paths (fallback)
    try:
        CURRENT_DIR = Path(__file__).parent.resolve()
    except NameError:
        CURRENT_DIR = Path.cwd()

    TOKENS_BIN_FILE = CURRENT_DIR / "pre_processing" / "marathi_tokens.bin"
    TOKENS_META_FILE = CURRENT_DIR / "pre_processing" / "marathi_tokens_meta.json"
    TOKENIZER_PATH = CURRENT_DIR / "tokenizer" / "marathi-bpe-tokenizer-49k.json"
    CHECKPOINT_DIR = CURRENT_DIR / "checkpoints"


In [16]:
# Training Hyperparameters
# Increased batch size for GPU (A100 can handle larger batches)
BATCH_SIZE = 32 if IN_COLAB else 4  # Larger batch for GPU
MAX_SEQ_LENGTH = 128  # Keep this small to start, increase later (e.g., 512 or 1024)
LEARNING_RATE = 1e-4
NUM_STEPS = 5000
CHECKPOINT_INTERVAL = 500
SAMPLE_INTERVAL = 100
SAMPLE_MAX_LENGTH = 50


In [9]:
class MemmapDataset(Dataset):
    """
    Memory-mapped dataset that reads pre-tokenized binary file.
    Uses numpy.memmap for efficient random access without loading entire file into RAM.
    """
    def __init__(self, bin_file, meta_file, max_length=128):
        """
        Args:
            bin_file: Path to the pre-tokenized binary file (marathi_tokens.bin)
            meta_file: Path to the metadata JSON file (marathi_tokens_meta.json)
            max_length: Maximum sequence length for training
        """
        self.max_length = max_length

        # Check if files exist
        if not os.path.exists(bin_file):
            raise FileNotFoundError(f"ERROR: Binary file not found at {bin_file}\n"
                                  f"Please run colab_ready_Pre_tok.py first to create the pre-tokenized file.")
        if not os.path.exists(meta_file):
            raise FileNotFoundError(f"ERROR: Metadata file not found at {meta_file}\n"
                                  f"Please run colab_ready_Pre_tok.py first to create the metadata file.")

        # Load metadata
        print(f"Loading metadata from {meta_file}...")
        with open(meta_file, 'r') as f:
            meta = json.load(f)

        self.vocab_size = meta['vocab_size']
        self.total_tokens = meta['total_tokens']
        dtype_str = meta['dtype']

        # Convert dtype string back to numpy dtype
        dtype_map = {
            'uint16': np.uint16,
            'uint32': np.uint32,
            'int32': np.int32
        }
        dtype = dtype_map.get(dtype_str, np.uint16)

        print(f"Metadata loaded: {self.total_tokens:,} tokens, vocab_size={self.vocab_size}, dtype={dtype_str}")

        # Memory-map the binary file (read-only, doesn't load into RAM)
        print(f"Memory-mapping binary file: {bin_file}...")
        print(f"   File size: {os.path.getsize(bin_file) / (1024*1024):.2f} MB")

        self.tokens = np.memmap(
            bin_file,
            dtype=dtype,
            mode='r',  # Read-only
            shape=(self.total_tokens,)
        )

        print(f"✅ Dataset loaded successfully using memory mapping")
        print(f"   Dataset size: {len(self):,} samples")
        print(f"   Memory footprint: ~{self.tokens.nbytes / (1024*1024):.2f} MB (virtual, not physical RAM)")

    def __len__(self):
        return max(0, len(self.tokens) - self.max_length)

    def __getitem__(self, idx):
        """Extract sequence starting at index idx."""
        chunk = self.tokens[idx:idx + self.max_length + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y


In [10]:
def get_latest_checkpoint(device):
    """Finds the latest checkpoint file, if one exists."""
    if not CHECKPOINT_DIR.exists():
        return None, 0

    checkpoints = list(CHECKPOINT_DIR.glob("checkpoint_*.pt"))
    if not checkpoints:
        return None, 0

    # Find the checkpoint with the highest step number
    latest_checkpoint = max(checkpoints, key=lambda x: int(x.stem.split('_')[1]))
    step = int(latest_checkpoint.stem.split('_')[1])

    try:
        torch.load(latest_checkpoint, map_location=device)
        print(f"Found latest checkpoint: {latest_checkpoint}")
        return str(latest_checkpoint), step
    except Exception as e:
        print(f"Could not load checkpoint {latest_checkpoint}: {e}")
        return None, 0


In [11]:
def save_checkpoint(model, optimizer, step, path):
    """Saves the model checkpoint."""
    # Create directory if it doesn't exist
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    # Full path for the new checkpoint
    checkpoint_path = CHECKPOINT_DIR / os.path.basename(path)

    # Delete old checkpoints to save space (keep only the latest)
    for f in CHECKPOINT_DIR.glob("checkpoint_*.pt"):
        if f != checkpoint_path:
            f.unlink()

    # Save new checkpoint
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, checkpoint_path)
    print(f"Checkpoint saved at step {step} to {checkpoint_path}")

    if IN_COLAB:
        print(f"   ✅ Saved to Google Drive (will persist after session ends)")


In [12]:
def generate_sample(model, tokenizer, device, prompt="आजचा दिवस", max_length=50):
    """Generates a text sample from the model."""
    print("\n--- Generating Sample ---")
    model.eval()

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(input_ids)
            next_token_logits = outputs[:, -1, :] / 1.0
            next_token = torch.argmax(next_token_logits, dim=-1)

            if next_token.item() == tokenizer.eos_token_id:
                break

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_text}\n")
    model.train()
    return generated_text


In [13]:
def check_gpu_setup():
    """Check and display GPU information."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        print(f"   CUDA Version: {torch.version.cuda}")
        return device
    else:
        print("⚠️  No GPU available. Training will be slower on CPU.")
        return torch.device("cpu")


In [14]:
def train(num_steps=5000, checkpoint_interval=500, learning_rate=1e-4):
    """Main training loop optimized for Colab with GPU support."""

    print("="*60)
    print("SmolLM-135M Marathi Training (Colab-Optimized)")
    print("="*60)

    if IN_COLAB:
        print("✅ Running in Google Colab")
        # Check if Drive is mounted (if using Drive paths)
        if str(TOKENS_BIN_FILE).startswith('/content/drive'):
            if not Path('/content/drive').exists():
                print("\n⚠️  WARNING: Google Drive not mounted!")
                print("   Please run: from google.colab import drive; drive.mount('/content/drive')")
                return None, None
    else:
        print("ℹ️  Running on local machine")

    # --- 0. GPU Setup ---
    print("\n" + "="*60)
    print("GPU Setup")
    print("="*60)
    device = check_gpu_setup()
    print(f"Using device: {device}")

    # --- 1. Training Configuration ---
    print("\n" + "="*60)
    print("Training Configuration")
    print("="*60)
    print(f"Training Steps: {num_steps:,}")
    print(f"Checkpoint Interval: {checkpoint_interval} steps")
    print(f"Learning Rate: {learning_rate}")
    print(f"Batch Size: {BATCH_SIZE} {'(GPU-optimized)' if device.type == 'cuda' else ''}")
    print(f"Max Sequence Length: {MAX_SEQ_LENGTH}")

    # --- 2. Initialize Model, Tokenizer, Config ---
    print("\n" + "="*60)
    print("Initializing Model and Tokenizer")
    print("="*60)

    print("Loading tokenizer...")
    try:
        tokenizer = PreTrainedTokenizerFast(
            tokenizer_file=str(TOKENIZER_PATH),
            bos_token="<s>",
            eos_token="</s>",
            pad_token="<pad>",
            unk_token="<unk>",
            mask_token="<mask>"
        )
        print(f"✅ Tokenizer loaded successfully")
    except Exception as e:
        print(f"ERROR: Could not load tokenizer from {TOKENIZER_PATH}")
        print(f"Details: {e}")
        if IN_COLAB:
            print(f"\n📝 Make sure tokenizer file is uploaded or in Google Drive")
            print(f"   Update TOKENIZER_PATH in the configuration section above")
        return None, None

    print("\nInitializing model...")
    config = SmolLMConfig()
    model = SmolLM2(config).to(device)
    print(f"✅ Model initialized and moved to {device}")

    # Critical Validation: Ensure vocab sizes match
    if config.vocab_size != tokenizer.vocab_size:
        print(f"\n❌ ERROR: Vocab size mismatch!")
        print(f"Model config.vocab_size is {config.vocab_size}")
        print(f"Tokenizer vocab_size is {tokenizer.vocab_size}")
        print("Please update SmolLMConfig in model.py to match your tokenizer.")
        return None, None

    # Print model details
    print("\n" + "="*60)
    print("Model Details")
    print("="*60)
    print(f"Model Architecture: SmolLM2 (Llama-style)")
    print(f"Hidden Size: {config.hidden_size}")
    print(f"Intermediate Size: {config.intermediate_size}")
    print(f"Number of Layers: {config.num_hidden_layers}")
    print(f"Number of Attention Heads: {config.num_attention_heads}")
    print(f"Number of Key-Value Heads: {config.num_key_value_heads}")
    print(f"Max Position Embeddings: {config.max_position_embeddings}")
    print(f"Vocabulary Size: {config.vocab_size:,}")

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

    param_size_mb = total_params * 4 / (1024 * 1024)
    print(f"Estimated Model Size (float32): {param_size_mb:.2f} MB")

    if device.type == 'cuda':
        print(f"\nGPU Memory Usage:")
        print(f"   Model: ~{param_size_mb:.2f} MB")
        print(f"   Training (with gradients): ~{param_size_mb * 3:.2f} MB")
        print(f"   Available GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("="*60 + "\n")

    # --- 3. Create Dataset and Dataloader ---
    print("="*60)
    print("Loading Dataset (Memory-Mapped)")
    print("="*60)
    try:
        dataset = MemmapDataset(TOKENS_BIN_FILE, TOKENS_META_FILE, max_length=MAX_SEQ_LENGTH)
        dataloader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=2 if device.type == 'cuda' else 0,  # Use multiple workers for GPU
            pin_memory=True if device.type == 'cuda' else False  # Faster GPU transfer
        )
        print(f"✅ Dataset loaded successfully")
        print(f"   Batch size: {BATCH_SIZE}")
        print(f"   Max sequence length: {MAX_SEQ_LENGTH}")
        print(f"   Total batches per epoch: {len(dataloader):,}")
        print(f"   Using memory-mapped access (minimal RAM usage)")
        if device.type == 'cuda':
            print(f"   DataLoader: {dataloader.num_workers} workers, pin_memory=True")
    except FileNotFoundError as e:
        print(f"❌ ERROR: {e}")
        print(f"\n📝 Setup Instructions:")
        if IN_COLAB:
            print(f"   1. Upload pre-tokenized files to Google Drive:")
            print(f"      - marathi_tokens.bin")
            print(f"      - marathi_tokens_meta.json")
            print(f"   2. Update TOKENS_BIN_FILE and TOKENS_META_FILE paths above")
            print(f"   3. Mount Google Drive: drive.mount('/content/drive')")
        else:
            print(f"   1. Run pre-tokenization script first:")
            print(f"      python colab_ready_Pre_tok.py  (for Colab)")
            print(f"      OR")
            print(f"      python new_sol.py  (for local machine)")
        return None, None
    except Exception as e:
        print(f"❌ ERROR: Failed to load dataset: {e}")
        return None, None

    # --- 4. Initialize Optimizer ---
    print("\nInitializing optimizer...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    print(f"✅ Optimizer initialized (AdamW, lr={learning_rate})")

    # --- 5. Load Checkpoint (if exists) ---
    print("\nChecking for existing checkpoints...")
    latest_checkpoint, initial_step = get_latest_checkpoint(device)
    if latest_checkpoint:
        print(f"📂 Found checkpoint at step {initial_step}")
        print(f"Loading checkpoint from {latest_checkpoint}...")
        checkpoint = torch.load(latest_checkpoint, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(f"✅ Resuming training from step {initial_step}")
    else:
        initial_step = 0
        print("🆕 No checkpoint found. Starting training from scratch.")

    # --- 6. Training Loop ---
    print("\n" + "="*60)
    print("Starting Training Loop")
    print("="*60)
    print(f"Target steps: {num_steps}")
    print(f"Starting from step: {initial_step}")
    print(f"Checkpoint interval: {checkpoint_interval} steps")
    print(f"Sample generation interval: {SAMPLE_INTERVAL} steps")
    if device.type == 'cuda':
        print(f"🚀 Training on GPU: {torch.cuda.get_device_name(0)}")
    print("="*60 + "\n")

    step = initial_step
    data_iter = iter(dataloader)
    model.train()

    # Track training speed
    import time
    start_time = time.time()

    while step < initial_step + num_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        # Move batch to device
        input_ids, labels = batch
        input_ids = input_ids.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward pass
        outputs = model(input_ids)

        # Calculate loss
        loss = F.cross_entropy(outputs.view(-1, config.vocab_size), labels.view(-1))

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # --- 7. Logging and Checkpointing ---
        if (step + 1) % 10 == 0:
            progress = ((step + 1 - initial_step) / num_steps) * 100
            elapsed = time.time() - start_time
            steps_per_sec = (step + 1 - initial_step) / elapsed if elapsed > 0 else 0
            print(f"Step {step + 1}/{initial_step + num_steps} ({progress:.1f}%) | "
                  f"Loss: {loss.item():.4f} | "
                  f"Speed: {steps_per_sec:.2f} steps/sec")

            if device.type == 'cuda' and (step + 1) % 100 == 0:
                # Show GPU memory usage
                gpu_mem = torch.cuda.memory_allocated(0) / 1e9
                gpu_mem_max = torch.cuda.max_memory_allocated(0) / 1e9
                print(f"   GPU Memory: {gpu_mem:.2f} GB (Peak: {gpu_mem_max:.2f} GB)")

        # Generate a sample
        if (step + 1) % SAMPLE_INTERVAL == 0:
            generate_sample(model, tokenizer, device, prompt="आजचा दिवस")

        # Save checkpoint
        if (step + 1) % checkpoint_interval == 0:
            checkpoint_path = f"checkpoint_{step + 1}.pt"
            save_checkpoint(model, optimizer, step + 1, checkpoint_path)

        step += 1

    print("\n" + "="*60)
    print(f"✅ Training completed at step {step}")
    print("="*60)

    # Final save
    print("\nSaving final checkpoint...")
    save_checkpoint(model, optimizer, step, "checkpoint_final.pt")
    print("✅ Final checkpoint saved!")

    if IN_COLAB:
        print(f"\n📁 Checkpoints saved to: {CHECKPOINT_DIR}")
        print("   These will persist in Google Drive after the session ends")

    return model, tokenizer


In [17]:
if __name__ == "__main__":
    print("="*60)
    print("SmolLM 135M Marathi Training (Colab-Optimized)")
    print("="*60)

    model, tokenizer = train(
        num_steps=NUM_STEPS,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        learning_rate=LEARNING_RATE
    )

    if model is not None and tokenizer is not None:
        print("\n" + "="*60)
        print("✅ Training run finished successfully!")
        print("="*60)
        if IN_COLAB:
            print("\n💡 Next steps:")
            print("   - Checkpoints are saved in Google Drive")
            print("   - You can download them or continue training in a new session")
            print("   - To download: files.download() or copy from Drive")
    else:
        print("\n" + "="*60)
        print("❌ Training run failed. Please check the errors above.")
        print("="*60)

SmolLM 135M Marathi Training (Colab-Optimized)
SmolLM-135M Marathi Training (Colab-Optimized)
✅ Running in Google Colab

GPU Setup
✅ GPU Available: NVIDIA A100-SXM4-80GB
   GPU Memory: 85.17 GB
   CUDA Version: 12.6
Using device: cuda

Training Configuration
Training Steps: 5,000
Checkpoint Interval: 500 steps
Learning Rate: 0.0001
Batch Size: 32 (GPU-optimized)
Max Sequence Length: 128

Initializing Model and Tokenizer
Loading tokenizer...
✅ Tokenizer loaded successfully

Initializing model...
✅ Model initialized and moved to cuda

Model Details
Model Architecture: SmolLM2 (Llama-style)
Hidden Size: 576
Intermediate Size: 1536
Number of Layers: 30
Number of Attention Heads: 9
Number of Key-Value Heads: 3
Max Position Embeddings: 2048
Vocabulary Size: 49,152
Total Parameters: 134,515,008
Trainable Parameters: 134,515,008
Estimated Model Size (float32): 513.13 MB

GPU Memory Usage:
   Model: ~513.13 MB
   Training (with gradients): ~1539.40 MB
   Available GPU Memory: 85.17 GB

Loading 